# E-Commerce Analytics System
# Notebook 3: Load Cleaned Data into SQLite

This notebook:
- Creates SQLite database
- Creates tables with constraints
- Loads cleaned CSV files
- Verifies row counts
- Checks referential integrity


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH=Path("database")
DB_PATH.mkdir(exist_ok=True)

db=DB_PATH/"ecommerce.db"

DATA=Path("data/cleaned")

customers=pd.read_csv(DATA/"customers_clean.csv")
products=pd.read_csv(DATA/"products_clean.csv")
orders=pd.read_csv(DATA/"orders_clean.csv")
order_items=pd.read_csv(DATA/"order_items_clean.csv")

conn=sqlite3.connect(db)
cursor=conn.cursor()


## Create Schema

In [ ]:
schema="""
PRAGMA foreign_keys=ON;

DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers(
customer_id INTEGER PRIMARY KEY,
customer_name TEXT NOT NULL,
email TEXT,
registration_date TEXT,
customer_type TEXT
);

CREATE TABLE products(
product_id INTEGER PRIMARY KEY,
product_name TEXT NOT NULL,
category TEXT,
subcategory TEXT,
cost_price REAL CHECK(cost_price>=0)
);

CREATE TABLE orders(
order_id INTEGER PRIMARY KEY,
customer_id INTEGER,
order_date TEXT,
status TEXT,
region_code TEXT,
FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items(
item_id INTEGER PRIMARY KEY,
order_id INTEGER,
product_id INTEGER,
quantity INTEGER,
unit_price REAL,
discount_percent REAL CHECK(discount_percent>=0 AND discount_percent<=100),
return_flag TEXT,
FOREIGN KEY(order_id) REFERENCES orders(order_id),
FOREIGN KEY(product_id) REFERENCES products(product_id)
);
"""
conn.executescript(schema)
print("Schema created.")


## Load Tables

In [ ]:
customers.to_sql("customers",conn,if_exists="append",index=False)
products.to_sql("products",conn,if_exists="append",index=False)

valid_orders=orders[orders["customer_id"].isin(customers["customer_id"]) | (orders["customer_id"]==-1)]
valid_orders.to_sql("orders",conn,if_exists="append",index=False)

valid_items=order_items[
order_items["order_id"].isin(valid_orders["order_id"]) &
order_items["product_id"].isin(products["product_id"])
]
valid_items.to_sql("order_items",conn,if_exists="append",index=False)

conn.commit()
print("Data loaded successfully.")


## Verification

In [ ]:
tables=["customers","products","orders","order_items"]

for t in tables:
    count=pd.read_sql(f"SELECT COUNT(*) AS rows FROM {t}",conn)
    print(f"\n{t.upper()}")
    print(count)


## Relationship Validation

In [ ]:
invalid_orders=pd.read_sql('''
SELECT oi.item_id,oi.order_id
FROM order_items oi
LEFT JOIN orders o
ON oi.order_id=o.order_id
WHERE o.order_id IS NULL;
''',conn)

invalid_products=pd.read_sql('''
SELECT oi.item_id,oi.product_id
FROM order_items oi
LEFT JOIN products p
ON oi.product_id=p.product_id
WHERE p.product_id IS NULL;
''',conn)

print("Broken Order FK :",len(invalid_orders))
print("Broken Product FK :",len(invalid_products))


## Preview Data

In [ ]:
query='''
SELECT
o.order_id,
c.customer_name,
p.product_name,
oi.quantity,
oi.unit_price,
o.status
FROM orders o
JOIN customers c
ON o.customer_id=c.customer_id
JOIN order_items oi
ON o.order_id=oi.order_id
JOIN products p
ON oi.product_id=p.product_id
LIMIT 10;
'''

pd.read_sql(query,conn)


## Close Connection

In [ ]:
conn.close()
print("SQLite database saved at:",db)
